# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [ ]:
import os
import sys
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import AzureOpenAI
from dotenv import load_dotenv

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    os.environ["AZURE_OPENAI_API_KEY"] = userdata.get("AZURE_OPENAI_API_KEY")
    os.environ["AZURE_OPENAI_API_VERSION"] = userdata.get("AZURE_OPENAI_API_VERSION")
    os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get("AZURE_OPENAI_ENDPOINT")
    os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = userdata.get("AZURE_OPENAI_CHAT_DEPLOYMENT")
else:
    load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)
chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
print("API version:", os.getenv("AZURE_OPENAI_API_VERSION"))
print("Deployment:", chat_deployment)
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

In [ ]:
DATA_DIR = "./data"
texts = []
source_files = []
for filename in os.listdir(DATA_DIR):
    if filename.endswith(".txt"):
        path = os.path.join(DATA_DIR, filename)

        with open(path, "r", encoding="utf-8") as f:
            file_text = f.read()

        texts.append(file_text)
        source_files.append(filename)

text = "\n\n".join(texts)
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]
if not chunks:
    raise ValueError("No text chunks found. Add .txt files to the ./data folder.")

print("Loaded files:")
for filename in source_files:
    print("-", filename)

print("\nChunks:")
for c in chunks:
    print("-", c)

In [ ]:
def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array([get_embedding(chunk) for chunk in chunks]).astype("float32")
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.reset()
index.add(chunk_embeddings)
print("FAISS index size:", index.ntotal)

In [ ]:
question = input("Enter your question: ")
print("Question:", question)
question_embedding = np.array([get_embedding(question)]).astype("float32")

In [ ]:
k = 3
distances, indices = index.search(question_embedding, k)
retrieved_chunks = [chunks[i] for i in indices[0]]
print("Retrieved context:")
for score, c in zip(distances[0], retrieved_chunks):
    print(f"[score={score:.4f}] - {c}")

In [ ]:
context = "\n\n".join(retrieved_chunks)
prompt = f"""
# Instructions
You are a Retrieval-Augmented Generation assistant.
Answer the question using only the provided context.
Do not use outside knowledge.
If the answer is not present in the context, say: "There is no information available in the knowledge base."
Keep the answer short and clear.

# Context:
{context}

# Question:
{question}

# Answer:
"""

In [ ]:
response = client.chat.completions.create(
    model=chat_deployment,
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)
print("\nLLM Answer:")
print(response.choices[0].message.content)